In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
trades = pd.read_csv('/content/historical_data (1).csv')
sentiment = pd.read_csv('/content/fear_greed_index (1).csv')

In [5]:
print("TRADES COLUMNS:", trades.columns)
print("SENTIMENT COLUMNS:", sentiment.columns)

TRADES COLUMNS: Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='object')
SENTIMENT COLUMNS: Index(['timestamp', 'value', 'classification', 'date'], dtype='object')


In [45]:
trades['Date'] = pd.to_datetime(trades['Timestamp IST'], format="%d-%m-%Y %H:%M")
sentiment['Date'] = pd.to_datetime(sentiment['date'])

In [8]:
data = pd.merge(trades, sentiment, on='Date', how='left')

print(data.head())

                                      Account  Coin  Execution Price  \
0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9769   
1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9800   
2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9855   
3  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9874   
4  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9894   

   Size Tokens  Size USD Side     Timestamp IST  Start Position Direction  \
0       986.87   7872.16  BUY  02-12-2024 22:50        0.000000       Buy   
1        16.00    127.68  BUY  02-12-2024 22:50      986.524596       Buy   
2       144.09   1150.63  BUY  02-12-2024 22:50     1002.518996       Buy   
3       142.98   1142.04  BUY  02-12-2024 22:50     1146.558564       Buy   
4         8.73     69.75  BUY  02-12-2024 22:50     1289.488521       Buy   

   Closed PnL  ...     Order ID  Crossed       Fee      Trade ID  \
0         0.0  ...  52017706630     

In [10]:
# Remove missing sentiment rows
data = data.dropna(subset=['classification'])

# Convert numeric columns
data['Closed PnL'] = pd.to_numeric(data['Closed PnL'], errors='coerce')

In [13]:
profit_by_sentiment = data.groupby('classification')['Closed PnL'].mean()
print(profit_by_sentiment)

Series([], Name: Closed PnL, dtype: float64)


In [32]:
if not profit_by_sentiment.empty:
    profit_by_sentiment.plot(kind='bar')
    plt.title('Average Profit by Market Sentiment')
    plt.xlabel('Sentiment')
    plt.ylabel('Average Profit')
    plt.show()
else:
    print("No data to plot for Average Profit by Market Sentiment. Please ensure 'data' DataFrame and 'profit_by_sentiment' are populated correctly from previous steps.")

No data to plot for Average Profit by Market Sentiment. Please ensure 'data' DataFrame and 'profit_by_sentiment' are populated correctly from previous steps.


In [29]:
trades['Date'] = pd.to_datetime(trades['Timestamp IST'], format="%d-%m-%Y %H:%M").dt.date
sentiment['Date'] = pd.to_datetime(sentiment['date']).dt.date

In [38]:
trade_count = data['classification'].value_counts()
print(trade_count)

if not trade_count.empty:
    trade_count.plot(kind='bar')
    plt.title('Number of Trades by Sentiment')
    plt.xlabel('Sentiment')
    plt.ylabel('Trades Count')
    plt.show()
else:
    print("No data to plot for Number of Trades by Sentiment. Please ensure 'data' DataFrame is populated correctly from previous steps.")

Series([], Name: count, dtype: int64)
No data to plot for Number of Trades by Sentiment. Please ensure 'data' DataFrame is populated correctly from previous steps.


In [30]:
data['Result'] = data['Closed PnL'].apply(lambda x: 'Win' if x > 0 else 'Loss')

win_rate = data.groupby('classification')['Result'].value_counts(normalize=True)
print(win_rate)

Series([], Name: proportion, dtype: float64)


In [39]:
print('Trades Date value counts (including NaT):\n', trades['Date'].value_counts(dropna=False))
print('\nSentiment Date value counts (including NaT):\n', sentiment['Date'].value_counts(dropna=False))

# Re-merge to check intermediate state
# This assumes R-duS0I0WZxF or WU0B5yO0ZhHC has been executed correctly
data_after_merge = pd.merge(trades, sentiment, on='Date', how='left')
print('\nData after merge (head):\n', data_after_merge.head())
print('\nNulls after merge:\n', data_after_merge.isnull().sum())

Trades Date value counts (including NaT):
 Date
2025-02-25    6246
2025-04-23    6159
2025-02-24    5616
2025-03-12    3968
2025-04-09    3967
              ... 
2024-10-04       1
2024-09-05       1
2024-09-08       1
2024-08-29       1
2024-07-08       1
Name: count, Length: 480, dtype: int64

Sentiment Date value counts (including NaT):
 Date
2025-05-02    1
2018-02-01    1
2018-02-02    1
2018-02-03    1
2025-04-16    1
             ..
2018-02-09    1
2018-02-08    1
2018-02-07    1
2018-02-06    1
2018-02-05    1
Name: count, Length: 2644, dtype: int64

Data after merge (head):
                                       Account  Coin  Execution Price  \
0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9769   
1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9800   
2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9855   
3  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9874   
4  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @1

In [47]:
# Check how many rows are missing sentiment
print(data['classification'].isna().sum(), "missing sentiment rows")

# Ensure 'Date' columns are datetime objects before merge_asof
# This re-confirms the type in case previous steps were not fully effective or a different dataframe was passed
trades['Date'] = pd.to_datetime(trades['Date'], errors='coerce')
sentiment['Date'] = pd.to_datetime(sentiment['Date'], errors='coerce')

# Fill missing sentiment using nearest date match
data = pd.merge_asof(
    trades.sort_values('Date'),
    sentiment.sort_values('Date'),
    on='Date',
    direction='nearest'
)

# Check again
print("New data shape:", data.shape)
data.head()

0 missing sentiment rows
New data shape: (211224, 21)


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,...,Order ID,Crossed,Fee,Trade ID,Timestamp,Date,timestamp,value,classification,date
0,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,ETH,1897.9,0.08240,156.39,BUY,01-05-2023 01:06,0.0967,Open Long,0.0,...,173271100,True,0.000000,0.000000e+00,1.680000e+12,2023-05-01 01:06:00,1682919000,63,Greed,2023-05-01
1,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,ETH,1898.6,0.07220,137.08,BUY,01-05-2023 01:06,0.1791,Open Long,0.0,...,173271100,True,0.000000,0.000000e+00,1.680000e+12,2023-05-01 01:06:00,1682919000,63,Greed,2023-05-01
2,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,ETH,1897.9,0.09670,183.53,BUY,01-05-2023 01:06,0.0000,Open Long,0.0,...,173271100,True,0.000000,0.000000e+00,1.680000e+12,2023-05-01 01:06:00,1682919000,63,Greed,2023-05-01
3,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,BTC,41866.0,0.58211,24370.62,SELL,05-12-2023 03:11,-0.0150,Open Short,0.0,...,4064974623,True,6.092654,2.830000e+14,1.700000e+12,2023-12-05 03:11:00,1701754200,75,Extreme Greed,2023-12-05
4,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,BTC,41867.0,0.01500,628.00,SELL,05-12-2023 03:11,0.0000,Open Short,0.0,...,4064974623,True,0.157001,1.070000e+15,1.700000e+12,2023-12-05 03:11:00,1701754200,75,Extreme Greed,2023-12-05


In [26]:
print("Trades Date value counts:\n", trades['Date'].value_counts())
print("Sentiment Date value counts:\n", sentiment['Date'].value_counts())
print("Trades Date min/max:", trades['Date'].min(), trades['Date'].max())
print("Sentiment Date min/max:", sentiment['Date'].min(), sentiment['Date'].max())

Trades Date value counts:
 Date
2025-02-25    6246
2025-04-23    6159
2025-02-24    5616
2025-03-12    3968
2025-04-09    3967
              ... 
2024-10-04       1
2024-09-05       1
2024-09-08       1
2024-08-29       1
2024-07-08       1
Name: count, Length: 480, dtype: int64
Sentiment Date value counts:
 Date
2025-05-02    1
2018-02-01    1
2018-02-02    1
2018-02-03    1
2025-04-16    1
             ..
2018-02-09    1
2018-02-08    1
2018-02-07    1
2018-02-06    1
2018-02-05    1
Name: count, Length: 2644, dtype: int64
Trades Date min/max: 2023-05-01 2025-05-01
Sentiment Date min/max: 2018-02-01 2025-05-02


In [33]:
print("Trades Date value counts:\n", trades['Date'].value_counts())
print("Sentiment Date value counts:\n", sentiment['Date'].value_counts())

print("Trades Date min/max:", trades['Date'].min(), trades['Date'].max())
print("Sentiment Date min/max:", sentiment['Date'].min(), sentiment['Date'].max())


Trades Date value counts:
 Date
2025-02-25    6246
2025-04-23    6159
2025-02-24    5616
2025-03-12    3968
2025-04-09    3967
              ... 
2024-10-04       1
2024-09-05       1
2024-09-08       1
2024-08-29       1
2024-07-08       1
Name: count, Length: 480, dtype: int64
Sentiment Date value counts:
 Date
2025-05-02    1
2018-02-01    1
2018-02-02    1
2018-02-03    1
2025-04-16    1
             ..
2018-02-09    1
2018-02-08    1
2018-02-07    1
2018-02-06    1
2018-02-05    1
Name: count, Length: 2644, dtype: int64
Trades Date min/max: 2023-05-01 2025-05-01
Sentiment Date min/max: 2018-02-01 2025-05-02


In [35]:
data['Result'] = data['Closed PnL'].apply(lambda x: 'Win' if x > 0 else 'Loss')

win_rate = data.groupby('classification')['Result'].value_counts(normalize=True)
print(win_rate)

Series([], Name: proportion, dtype: float64)


In [48]:
print(data.shape)
data.head()

(211224, 21)


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,...,Order ID,Crossed,Fee,Trade ID,Timestamp,Date,timestamp,value,classification,date
0,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,ETH,1897.9,0.08240,156.39,BUY,01-05-2023 01:06,0.0967,Open Long,0.0,...,173271100,True,0.000000,0.000000e+00,1.680000e+12,2023-05-01 01:06:00,1682919000,63,Greed,2023-05-01
1,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,ETH,1898.6,0.07220,137.08,BUY,01-05-2023 01:06,0.1791,Open Long,0.0,...,173271100,True,0.000000,0.000000e+00,1.680000e+12,2023-05-01 01:06:00,1682919000,63,Greed,2023-05-01
2,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,ETH,1897.9,0.09670,183.53,BUY,01-05-2023 01:06,0.0000,Open Long,0.0,...,173271100,True,0.000000,0.000000e+00,1.680000e+12,2023-05-01 01:06:00,1682919000,63,Greed,2023-05-01
3,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,BTC,41866.0,0.58211,24370.62,SELL,05-12-2023 03:11,-0.0150,Open Short,0.0,...,4064974623,True,6.092654,2.830000e+14,1.700000e+12,2023-12-05 03:11:00,1701754200,75,Extreme Greed,2023-12-05
4,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,BTC,41867.0,0.01500,628.00,SELL,05-12-2023 03:11,0.0000,Open Short,0.0,...,4064974623,True,0.157001,1.070000e+15,1.700000e+12,2023-12-05 03:11:00,1701754200,75,Extreme Greed,2023-12-05


In [49]:
print(data.shape)

(211224, 21)


In [50]:
print("Trades date range:", trades['Date'].min(), trades['Date'].max())
print("Sentiment date range:", sentiment['Date'].min(), sentiment['Date'].max())

Trades date range: 2023-05-01 01:06:00 2025-05-01 12:13:00
Sentiment date range: 2018-02-01 00:00:00 2025-05-02 00:00:00
